In [23]:
import torch
import torch.nn as nn
import numpy as np

### クロスエントロピー誤差計算前のlogitとラベル

In [75]:
pred = torch.tensor([
    [0.1, 0.1, 0.8],
    [0.1, 0.5, 0.3],
    [0, 0, 1],
])

label = torch.tensor([
    0, 1, 2
])

In [11]:
torch.arange(len(pred))

tensor([0, 1, 2])

In [56]:
x = pred[torch.arange(len(pred)), label]
x

tensor([0.1000, 0.5000, 1.0000])

### 普通のクロスエントロピー誤差

### 計算① pytorchでのクロスエントロピー誤差計算

In [70]:
loss = nn.CrossEntropyLoss()

In [71]:
loss(pred, label).item()

0.9510242938995361

### 計算② マニュアルでの計算

In [76]:
def softmax(x):
    if x.ndim == 2:
        x_max = x.max(axis=1).values.reshape(-1, 1)
        x = x - x_max
        x_exp = torch.exp(x)
        z = x_exp / x_exp.sum(axis=1).reshape(-1, 1)
    
    elif x.ndim == 1:
        x_max = x.max().values
        x = x - x_max
        x_exp = torch.exp(x)
        z = x_exp / x_exp.sum()  
    return z

In [77]:
z = softmax(pred)
z

tensor([[0.2491, 0.2491, 0.5017],
        [0.2693, 0.4018, 0.3289],
        [0.2119, 0.2119, 0.5761]])

In [78]:
y = -torch.log(z[torch.arange(len(z)), label])
y.mean()

tensor(0.9510)

### ignore indexを使った計算

>ignore_index (int, 省略可) – 無視され、入力の勾配計算に寄与しないターゲット値を指定します。size_average が True の場合、損失は無視されていないターゲットに対して平均化されます。なお、ignore_index は、ターゲットにクラスインデックスが含まれている場合にのみ適用可能です。

- 無視したいラベルを除外して
- 残ったデータ数で平均する

In [117]:
ignore_index = 1

### 計算① pytorchでのクロスエントロピー誤差計算

In [107]:
loss = nn.CrossEntropyLoss(ignore_index=ignore_index)
loss(pred, label).item()

0.9705857038497925

### 計算② マニュアルでの計算

In [108]:
mask = label != ignore_index
mask

tensor([ True, False,  True])

In [113]:
ignore_y = softmax(pred)
ignore_y = -torch.log(ignore_y [torch.arange(len(pred)), label])
ignore_y

tensor([1.3897, 0.9119, 0.5514])

In [114]:
ignore_y *= mask
ignore_y

tensor([1.3897, 0.0000, 0.5514])

In [115]:
ignore_y.sum(), mask.sum()

(tensor(1.9412), tensor(2))

In [ ]:
l = ignore_y.sum() / mask.sum() # 有効な値でのみ平均化
l

tensor(0.9706)